# Ejemplo programable: agente recomendador de movilidad

Este notebook convierte la especificación conceptual de la Semana 1 en un agente de reglas sencillo y auditable. El agente usa una **reproducción histórica** de viajes Yellow Taxi para recomendar qué casos debería revisar una persona responsable. No controla vehículos ni demuestra que una recomendación reduzca esperas.

**Objetivos de aprendizaje**

- distinguir entorno, sensor lógico, percepción, estado interno y acción;
- implementar una función de agente como programa Python;
- conectar el código con PEAS, racionalidad limitada y propiedades del ambiente;
- aplicar abstención segura, capacidad simulada y supervisión humana;
- impedir que datos futuros entren en una decisión pasada.

## 1. Alcance y límites de interpretación

NYC TLC registra viajes realizados y reportados. No contiene demanda total, solicitudes no atendidas, tiempos de espera ni vehículos libres. NOAA aporta observaciones de una estación meteorológica, no el clima exacto de cada zona.

Los archivos son históricos y se publican después de los eventos. Para practicar el ciclo percepción-acción adoptamos una convención didáctica:

$$
\text{datos revelados hasta el cierre de }h
\longrightarrow
\text{recomendación simulada para }h+1.
$$

El agente solo emite mensajes. `RECOMENDAR_REFUERZO` significa que una persona debería considerar priorizar capacidad simulada; no significa que exista un vehículo libre ni que el agente lo haya trasladado.

In [ ]:
from dataclasses import asdict, dataclass
from io import StringIO
from urllib.request import urlopen

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 2. Períodos del experimento

La primera semana se usa como referencia histórica y queda cerrada antes del replay. La segunda semana se revela hora por hora. Así, los cuantiles de referencia no se ajustan con información futura del período evaluado.

In [ ]:
ZONA_HORARIA = "America/New_York"
INICIO_REFERENCIA = pd.Timestamp("2024-01-01 00:00:00", tz=ZONA_HORARIA)
FIN_REFERENCIA = pd.Timestamp("2024-01-08 00:00:00", tz=ZONA_HORARIA)
FIN_REPLAY = pd.Timestamp("2024-01-15 00:00:00", tz=ZONA_HORARIA)

URL_VIAJES = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
URL_ZONAS = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
URL_CLIMA = (
    "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/"
    "hourly/access/by-year/2024/psv/GHCNh_USW00094728_2024.psv"
)

print("Referencia:", INICIO_REFERENCIA, "→", FIN_REFERENCIA)
print("Replay:     ", FIN_REFERENCIA, "→", FIN_REPLAY)

## 3. Adquisición y calidad de los viajes

Leemos únicamente dos semanas y las columnas necesarias. Los filtros de calidad son criterios analíticos transparentes; no convierten los registros en una medición de demanda completa.

In [ ]:
COLUMNAS_VIAJES = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "PULocationID",
    "DOLocationID", "fare_amount", "total_amount",
]
filtros = [
    ("tpep_pickup_datetime", ">=", INICIO_REFERENCIA.tz_localize(None).to_pydatetime()),
    ("tpep_pickup_datetime", "<", FIN_REPLAY.tz_localize(None).to_pydatetime()),
]
viajes_originales = pd.read_parquet(
    URL_VIAJES, columns=COLUMNAS_VIAJES, filters=filtros, engine="pyarrow"
)
print(f"Viajes cargados: {len(viajes_originales):,}")

In [ ]:
calidad = viajes_originales.copy()
calidad["duracion_min"] = (
    calidad["tpep_dropoff_datetime"] - calidad["tpep_pickup_datetime"]
).dt.total_seconds() / 60
calidad["ok_pickup"] = calidad["tpep_pickup_datetime"].notna()
calidad["ok_dropoff"] = calidad["tpep_dropoff_datetime"].notna()
calidad["ok_duracion"] = calidad["duracion_min"].gt(0) & calidad["duracion_min"].le(24 * 60)
calidad["ok_distancia"] = calidad["trip_distance"].between(0, 100, inclusive="both")
calidad["ok_zonas"] = calidad[["PULocationID", "DOLocationID"]].notna().all(axis=1)
calidad["ok_importes"] = (
    calidad["fare_amount"].between(0, 1_000, inclusive="both")
    & calidad["total_amount"].between(0, 1_000, inclusive="both")
    & calidad["total_amount"].ge(calidad["fare_amount"])
)
calidad["ok_pasajeros"] = calidad["passenger_count"].isna() | calidad["passenger_count"].between(0, 8)
REGLAS_CALIDAD = [columna for columna in calidad if columna.startswith("ok_")]
calidad["registro_valido"] = calidad[REGLAS_CALIDAD].all(axis=1)
viajes = calidad.loc[calidad["registro_valido"]].copy()
viajes["pickup_hora"] = (
    viajes["tpep_pickup_datetime"]
    .dt.tz_localize(ZONA_HORARIA, ambiguous="raise", nonexistent="raise")
    .dt.floor("h")
)
print(f"Válidos: {len(viajes):,} | Rechazados: {len(calidad) - len(viajes):,}")

## 4. Percepción básica: actividad por zona-hora

El catálogo traduce `PULocationID` a una zona interpretable. Después construimos una grilla completa: una hora sin filas debe representar cero pickups reportados, no desaparecer del historial. Cada fila sigue siendo una representación parcial de actividad Yellow Taxi.

In [ ]:
zonas = pd.read_csv(URL_ZONAS).rename(
    columns={"LocationID": "PULocationID", "Borough": "borough", "Zone": "zona"}
)
zonas["PULocationID"] = pd.to_numeric(zonas["PULocationID"], errors="raise").astype("int64")
if zonas["PULocationID"].duplicated().any():
    raise ValueError("El catálogo de zonas no tiene una clave única")

conteos = (
    viajes.groupby(["pickup_hora", "PULocationID"], observed=True)
    .size().rename("pickups").reset_index()
)
horas = pd.date_range(INICIO_REFERENCIA, FIN_REPLAY, freq="h", inclusive="left")
grilla = pd.MultiIndex.from_product(
    [horas, zonas["PULocationID"]], names=["pickup_hora", "PULocationID"]
).to_frame(index=False)
actividad = (
    grilla.merge(conteos, on=["pickup_hora", "PULocationID"], how="left", validate="one_to_one")
    .merge(zonas[["PULocationID", "borough", "zona"]], on="PULocationID", how="left", validate="many_to_one")
)
actividad["pickups"] = actividad["pickups"].fillna(0).astype("int64")
actividad["hora_dia"] = actividad["pickup_hora"].dt.hour
assert not actividad.duplicated(["pickup_hora", "PULocationID"]).any()
print(f"Filas zona-hora: {len(actividad):,}")
display(actividad.head())

## 5. Contexto climático admisible

El sensor lógico resume NOAA a una observación por hora local y la une como contexto. La política predeterminada **no usa el clima para elevar prioridad**. Esta decisión requiere distinguir adquisición, tiempo, cobertura espacial y uso de la información.

### ¿Qué es el sensor lógico?

No es un instrumento meteorológico físico ni una columna aislada del archivo. Es el mecanismo de software que descarga los registros NOAA, comprueba que proceden de la estación `USW00094728`, interpreta `DATE` como UTC, convierte las marcas a la hora local de Nueva York, filtra el período del experimento, transforma las variables a valores numéricos y detecta faltantes. La **percepción climática** es el resultado validado que ese mecanismo entrega al agente: `temperatura_c`, `precipitacion_mm`, la hora correspondiente y un indicador de disponibilidad.

### ¿Qué significa una observación por hora local?

GHCNh puede contener varias muestras dentro de una misma hora. El código agrupa todas las muestras que pertenecen a la misma hora local y construye una fila resumida:

- `temperatura_c` es la media de las temperaturas numéricas disponibles;
- `precipitacion_mm` es la suma de las observaciones de precipitación;
- `min_count=1` evita convertir una hora completamente faltante en una precipitación artificial de cero.

Por tanto, "una observación por hora" significa **una fila horaria agregada**, no necesariamente una única lectura del instrumento. La conversión a `America/New_York` es necesaria porque los viajes TLC se organizan en hora local y NOAA registra las fechas en UTC. Sin esa conversión se podrían unir eventos que ocurrieron en horas distintas.

### ¿Por qué se llama contexto climático admisible?

Una variable es admisible si estaba disponible en el instante simulado de decisión. Para recomendar sobre la hora $h+1$, el replay solo puede revelar información hasta el cierre de $h$. Bajo la convención didáctica del notebook, al terminar las 08:00 se puede incorporar el resumen meteorológico de las 08:00 y recomendar para las 09:00. No se puede leer la observación que realmente ocurrió a las 09:00 y presentarla como si se hubiera conocido una hora antes. Eso sería **fuga de información futura**.

```text
clima observado durante h ── disponible al cerrar h ──► contexto para decidir sobre h+1
clima observado durante h+1 ──────────────────────────► todavía no disponible
```

Esta disponibilidad horaria pertenece al replay. Los archivos históricos NOAA y TLC usados en el curso no constituyen un flujo operacional en tiempo real. El clima de $h+1$ solo sería admisible si procediera de un **pronóstico emitido y disponible antes del cierre de $h$**, conservando además su hora de emisión y versión.

### ¿Qué significa unir el clima como contexto?

`clima_horario` tiene como máximo una fila por hora, mientras que `actividad` contiene una fila por zona y hora. La unión `many_to_one` copia el mismo resumen de la estación en todas las zonas de esa hora:

| zona | hora | pickups | temperatura NOAA |
|---|---|---:|---:|
| A | 08:00 | 20 | 2,2 °C |
| B | 08:00 | 35 | 2,2 °C |
| C | 08:00 | 8 | 2,2 °C |

Los tres valores de `2,2 °C` son copias de una misma observación horaria. No representan tres mediciones zonales independientes. El dato puede acompañar la explicación, permitir una auditoría y advertir sobre condiciones generales, pero no permite afirmar que esa era la temperatura exacta en cada zona TLC.

### ¿Por qué el clima no eleva automáticamente la prioridad?

1. **Razón temporal:** el clima observado en $h$ no garantiza el clima de $h+1$. Suponer persistencia sin un modelo o pronóstico convertiría una posibilidad en certeza.
2. **Razón espacial:** una estación puntual no describe toda la variación meteorológica de Nueva York. La distancia, la costa, la urbanización y otros factores pueden producir diferencias entre zonas.
3. **Razón inferencial:** observar lluvia, frío o una asociación con pickups no demuestra que el clima cause una necesidad de refuerzo. Los pickups también dependen de hora, oferta de taxis, eventos, tráfico y prácticas de reporte.
4. **Razón operativa:** todavía no existe una regla climática validada que indique qué variable usar, con qué umbral, durante cuánto tiempo y con qué beneficio esperado. Elevar prioridad sin esa especificación haría menos auditable la política.

En la política actual, la prioridad `BAJA`, `MEDIA` o `ALTA` depende únicamente de los pickups reportados y de su referencia histórica. El clima permanece en la percepción y en la bitácora para interpretar y auditar el caso, pero no modifica esos niveles.

### Diferencia entre exigir clima y usarlo para elevar prioridad

La configuración `requiere_clima=False` significa que un faltante meteorológico no impide decidir con la evidencia de actividad. Si se cambia a `requiere_clima=True`, la falta de temperatura y precipitación vuelve insuficiente la percepción y el agente debe `ABSTENERSE`. Incluso en ese caso, **tener clima disponible no aumenta la prioridad**: solo satisface una precondición de calidad.

Para que el clima influyera legítimamente en la acción harían falta, como mínimo, pronósticos disponibles al corte, cobertura espacial adecuada, umbrales definidos antes del replay, tratamiento explícito de incertidumbre y faltantes, validación histórica fuera de muestra y una evaluación que demuestre que la nueva regla mejora la función de desempeño sin producir efectos desiguales.

In [ ]:
COLUMNAS_CLIMA = ["STATION", "DATE", "temperature", "precipitation"]
with urlopen(URL_CLIMA) as respuesta:
    texto_clima = respuesta.read().decode("utf-8")
clima = pd.read_csv(StringIO(texto_clima), sep="|", usecols=COLUMNAS_CLIMA, low_memory=False)
if not clima["STATION"].astype(str).eq("USW00094728").all():
    raise ValueError("La fuente climática no corresponde a la estación declarada")
clima["fecha_ny"] = pd.to_datetime(clima["DATE"], utc=True, errors="coerce").dt.tz_convert(ZONA_HORARIA)
clima = clima.loc[clima["fecha_ny"].between(INICIO_REFERENCIA, FIN_REPLAY, inclusive="left")].copy()
clima[["temperature", "precipitation"]] = clima[["temperature", "precipitation"]].apply(pd.to_numeric, errors="coerce")
clima["pickup_hora"] = clima["fecha_ny"].dt.floor("h")
clima_horario = (
    clima.groupby("pickup_hora", as_index=False)
    .agg(
        temperatura_c=("temperature", "mean"),
        precipitacion_mm=("precipitation", lambda serie: serie.sum(min_count=1)),
    )
)
actividad = actividad.merge(clima_horario, on="pickup_hora", how="left", validate="many_to_one")
assert not actividad.duplicated(["pickup_hora", "PULocationID"]).any()
print("Horas climáticas disponibles:", clima_horario["pickup_hora"].nunique())

## 6. Referencia fijada antes del replay

Para cada zona y hora del día calculamos mediana y percentil 75 usando solo la primera semana. Con siete días hay siete referencias por combinación zona-hora; es suficiente para el ejercicio, pero insuficiente para una política operativa robusta.

In [ ]:
datos_referencia = actividad.loc[actividad["pickup_hora"] < FIN_REFERENCIA].copy()
referencia = (
    datos_referencia.groupby(["PULocationID", "hora_dia"], as_index=False, observed=True)
    .agg(
        n_referencias=("pickups", "size"),
        mediana_pickups=("pickups", "median"),
        p75_pickups=("pickups", lambda serie: serie.quantile(0.75)),
    )
)
assert datos_referencia["pickup_hora"].max() < FIN_REFERENCIA
assert referencia["n_referencias"].eq(7).all()
display(referencia.head())

## 7. PEAS que implementará el programa

| Componente | Implementación didáctica |
|---|---|
| **Performance** | Coherencia entre prioridad y acción, abstención correcta, cumplimiento de capacidad y trazabilidad |
| **Environment** | Replay histórico, zonas TLC, actividad reportada, estación NOAA, política y responsable humano |
| **Actuators** | Mensajes `OBSERVAR`, `MONITOREAR`, `RECOMENDAR_REFUERZO`; `ABSTENERSE` como falla segura |
| **Sensors** | Funciones que revelan la hora cerrada, validan datos y construyen la percepción; las columnas no son sensores por sí solas |

El entorno urbano real es parcialmente observable, estocástico, secuencial, dinámico, mixto y multiagente. El programa de reglas es determinista para una entrada congelada; esa propiedad del prototipo no vuelve determinista a la movilidad real.

## 8. Política y programa del agente

Los umbrales se declaran antes de ejecutar el replay. La prioridad es un **estado interno**, no una verdad sobre demanda. Entre los casos altos, la capacidad simulada se asigna primero al mayor excedente sobre el percentil 75 y luego por identificador de zona para resolver empates de forma reproducible.

In [ ]:
@dataclass(frozen=True)
class PoliticaAgente:
    minimo_referencias: int = 5
    capacidad_por_hora: int = 5
    requiere_clima: bool = False
    version: str = "1.0"


POLITICA = PoliticaAgente()
display(pd.Series(asdict(POLITICA), name="valor").to_frame())

In [ ]:
class AgenteRecomendadorMovilidad:
    ACCIONES = {"OBSERVAR", "MONITOREAR", "RECOMENDAR_REFUERZO", "ABSTENERSE"}

    def __init__(self, actividad_zona_hora, referencia_historica, politica):
        self.actividad = actividad_zona_hora.copy()
        self.referencia = referencia_historica.copy()
        self.politica = politica
        if self.actividad.duplicated(["pickup_hora", "PULocationID"]).any():
            raise ValueError("La percepción potencial no tiene clave zona-hora única")

    def percibir(self, hora_cerrada):
        hora = pd.Timestamp(hora_cerrada)
        hora = hora.tz_localize(ZONA_HORARIA) if hora.tzinfo is None else hora.tz_convert(ZONA_HORARIA)
        if hora < FIN_REFERENCIA or hora >= FIN_REPLAY:
            raise ValueError("La hora cerrada debe pertenecer al período de replay")
        percepcion = self.actividad.loc[self.actividad["pickup_hora"].eq(hora)].copy()
        if percepcion.empty:
            raise ValueError("La hora solicitada no está disponible en el replay")
        percepcion = percepcion.merge(
            self.referencia, on=["PULocationID", "hora_dia"], how="left", validate="many_to_one"
        )
        percepcion["hora_cerrada"] = hora
        percepcion["hora_objetivo"] = hora + pd.Timedelta(hours=1)
        percepcion["modo_replay"] = True
        percepcion["fuente_actividad"] = "NYC TLC Yellow Taxi histórico"
        percepcion["fuente_clima"] = "NOAA USW00094728"
        percepcion["clima_disponible"] = percepcion[["temperatura_c", "precipitacion_mm"]].notna().any(axis=1)
        percepcion["evidencia_suficiente"] = (
            percepcion["zona"].notna()
            & percepcion["n_referencias"].ge(self.politica.minimo_referencias)
            & percepcion[["mediana_pickups", "p75_pickups"]].notna().all(axis=1)
        )
        if self.politica.requiere_clima:
            percepcion["evidencia_suficiente"] &= percepcion["clima_disponible"]
        return percepcion

    def actualizar_estado(self, percepcion):
        estado = percepcion.copy()
        condiciones = [
            ~estado["evidencia_suficiente"],
            estado["pickups"].le(estado["mediana_pickups"]),
            estado["pickups"].le(estado["p75_pickups"]),
        ]
        estado["prioridad_interna"] = np.select(
            condiciones, ["NO_CONFIABLE", "BAJA", "MEDIA"], default="ALTA"
        )
        estado["exceso_sobre_p75"] = estado["pickups"] - estado["p75_pickups"]
        return estado

    def decidir(self, estado, capacidad_simulada):
        if capacidad_simulada is not None:
            if isinstance(capacidad_simulada, bool) or not isinstance(capacidad_simulada, (int, np.integer)):
                raise TypeError("La capacidad debe ser un entero no negativo o None")
            if capacidad_simulada < 0:
                raise ValueError("La capacidad no puede ser negativa")

        resultado = estado.sort_values(
            ["exceso_sobre_p75", "PULocationID"], ascending=[False, True], na_position="last"
        ).copy()
        capacidad = capacidad_simulada
        acciones, motivos, antes, despues = [], [], [], []
        for fila in resultado.itertuples():
            antes.append(capacidad)
            if fila.prioridad_interna == "NO_CONFIABLE":
                accion, motivo = "ABSTENERSE", "Evidencia insuficiente o inválida"
            elif fila.prioridad_interna == "BAJA":
                accion, motivo = "OBSERVAR", "Actividad no supera la mediana histórica fijada"
            elif fila.prioridad_interna == "MEDIA":
                accion, motivo = "MONITOREAR", "Actividad entre la mediana y el percentil 75"
            elif capacidad is None:
                accion, motivo = "ABSTENERSE", "No se conoce una precondición crítica: capacidad simulada"
            elif capacidad > 0:
                accion, motivo = "RECOMENDAR_REFUERZO", "Prioridad alta y capacidad simulada disponible"
                capacidad -= 1
            else:
                accion, motivo = "MONITOREAR", "Prioridad alta, pero la capacidad simulada está agotada"
            acciones.append(accion)
            motivos.append(motivo)
            despues.append(capacidad)

        resultado["accion"] = acciones
        resultado["motivo"] = motivos
        resultado["capacidad_antes"] = antes
        resultado["capacidad_despues"] = despues
        resultado["version_politica"] = self.politica.version
        resultado["decision_humana"] = "PENDIENTE"
        resultado["limitacion"] = (
            "Pickups TLC son viajes reportados, no demanda total; la capacidad es simulada y NOAA es una estación."
        )
        return resultado.sort_values("PULocationID").reset_index(drop=True)

    def recomendar(self, hora_cerrada, capacidad_simulada=None):
        percepcion = self.percibir(hora_cerrada)
        estado = self.actualizar_estado(percepcion)
        return self.decidir(estado, capacidad_simulada)


agente = AgenteRecomendadorMovilidad(actividad, referencia, POLITICA)

## 9. Un ciclo percepción-acción

Seleccionamos automáticamente una hora del replay que contenga casos por encima de su referencia. La capacidad es un parámetro externo del escenario y no procede del archivo TLC.

In [ ]:
candidatos_replay = (
    actividad.loc[actividad["pickup_hora"].between(FIN_REFERENCIA, FIN_REPLAY, inclusive="left")]
    .merge(referencia, on=["PULocationID", "hora_dia"], how="left", validate="many_to_one")
)
candidatos_altos = candidatos_replay.loc[candidatos_replay["pickups"].gt(candidatos_replay["p75_pickups"])]
if candidatos_altos.empty:
    raise RuntimeError("No se encontró un caso alto para la demostración")
hora_demo = candidatos_altos.sort_values(["pickup_hora", "PULocationID"]).iloc[0]["pickup_hora"]
recomendaciones_demo = agente.recomendar(hora_demo, capacidad_simulada=POLITICA.capacidad_por_hora)
print("Hora cerrada:", hora_demo, "| Hora objetivo:", hora_demo + pd.Timedelta(hours=1))
display(recomendaciones_demo.loc[recomendaciones_demo["accion"].ne("OBSERVAR"), [
    "PULocationID", "borough", "zona", "pickups", "mediana_pickups", "p75_pickups",
    "temperatura_c", "prioridad_interna", "accion", "motivo",
]].head(12))

## 10. Escenarios de prueba

Comparamos el mismo estado con y sin capacidad. Después simulamos una falla de percepción. La racionalidad se evalúa con la información disponible y las restricciones del escenario, no por conocer el resultado futuro.

In [ ]:
con_capacidad = agente.recomendar(hora_demo, capacidad_simulada=1)
sin_capacidad = agente.recomendar(hora_demo, capacidad_simulada=0)
zona_alta = con_capacidad.loc[con_capacidad["accion"].eq("RECOMENDAR_REFUERZO"), "PULocationID"].iloc[0]
comparacion = pd.concat([
    con_capacidad.loc[con_capacidad["PULocationID"].eq(zona_alta)].assign(escenario="capacidad disponible"),
    sin_capacidad.loc[sin_capacidad["PULocationID"].eq(zona_alta)].assign(escenario="capacidad agotada"),
])
display(comparacion[["escenario", "zona", "prioridad_interna", "accion", "motivo"]])

In [ ]:
percepcion_fallida = agente.percibir(hora_demo).head(1).copy()
percepcion_fallida["zona"] = pd.NA
percepcion_fallida["evidencia_suficiente"] = False
estado_fallido = agente.actualizar_estado(percepcion_fallida)
salida_fallida = agente.decidir(estado_fallido, capacidad_simulada=1)
display(salida_fallida[["PULocationID", "zona", "prioridad_interna", "accion", "motivo"]])
assert salida_fallida.loc[0, "accion"] == "ABSTENERSE"

## 11. Prueba de causalidad temporal

El archivo completo está físicamente cargado, pero el sensor lógico solo revela la hora cerrada. Alteramos artificialmente todos los pickups posteriores a la decisión y comprobamos que la salida anterior permanece idéntica.

In [ ]:
actividad_con_futuro_alterado = actividad.copy()
actividad_con_futuro_alterado.loc[
    actividad_con_futuro_alterado["pickup_hora"].gt(hora_demo), "pickups"
] += 10_000
agente_control = AgenteRecomendadorMovilidad(actividad_con_futuro_alterado, referencia, POLITICA)
salida_original = agente.recomendar(hora_demo, capacidad_simulada=2).sort_values("PULocationID")
salida_control = agente_control.recomendar(hora_demo, capacidad_simulada=2).sort_values("PULocationID")
COLUMNAS_DECISION = ["PULocationID", "pickups", "prioridad_interna", "accion", "capacidad_despues"]
pd.testing.assert_frame_equal(
    salida_original[COLUMNAS_DECISION].reset_index(drop=True),
    salida_control[COLUMNAS_DECISION].reset_index(drop=True),
)
print("Prueba temporal: OK. Cambiar el futuro no alteró la decisión en h.")

## 12. Replay de un día y evaluación interna

Ejecutamos 24 ciclos. Las métricas siguientes evalúan la coherencia del programa, no resultados de movilidad. No podemos medir reducción de espera, demanda no atendida ni beneficio causal con estas fuentes.

In [ ]:
horas_dia = pd.date_range(FIN_REFERENCIA, FIN_REFERENCIA + pd.Timedelta(days=1), freq="h", inclusive="left")
bitacora = pd.concat(
    [agente.recomendar(hora, capacidad_simulada=POLITICA.capacidad_por_hora) for hora in horas_dia],
    ignore_index=True,
)
display(bitacora["accion"].value_counts().rename_axis("accion").to_frame("casos"))
display(bitacora.loc[bitacora["accion"].eq("RECOMENDAR_REFUERZO"), [
    "hora_cerrada", "hora_objetivo", "zona", "pickups", "prioridad_interna",
    "accion", "decision_humana",
]].head(10))

In [ ]:
assert bitacora["accion"].isin(agente.ACCIONES).all()
assert bitacora["hora_objetivo"].eq(bitacora["hora_cerrada"] + pd.Timedelta(hours=1)).all()
assert bitacora["modo_replay"].all()
assert bitacora.loc[bitacora["accion"].eq("RECOMENDAR_REFUERZO"), "prioridad_interna"].eq("ALTA").all()
assert bitacora.loc[bitacora["prioridad_interna"].eq("NO_CONFIABLE"), "accion"].eq("ABSTENERSE").all()
refuerzos_por_hora = bitacora["accion"].eq("RECOMENDAR_REFUERZO").groupby(bitacora["hora_cerrada"]).sum()
assert refuerzos_por_hora.le(POLITICA.capacidad_por_hora).all()
assert bitacora["decision_humana"].eq("PENDIENTE").all()

coherencia = pd.Series({
    "acciones_válidas": bitacora["accion"].isin(agente.ACCIONES).mean(),
    "refuerzos_con_prioridad_alta": bitacora.loc[bitacora["accion"].eq("RECOMENDAR_REFUERZO"), "prioridad_interna"].eq("ALTA").mean(),
    "horas_dentro_de_capacidad": refuerzos_por_hora.le(POLITICA.capacidad_por_hora).mean(),
})
display(coherencia.rename("proporción").to_frame())
print("Invariantes de política: OK")

## 13. Lectura crítica

El programa es racional **respecto de la política declarada** si aplica consistentemente prioridades, capacidad y abstención. Eso no demuestra que la política sea social u operacionalmente adecuada. Elegir zonas con mayor actividad histórica puede reproducir dónde hubo oferta y viajes reportados, invisibilizar solicitudes no atendidas y concentrar atención en zonas ya servidas.

La salida conserva `decision_humana = PENDIENTE`: una persona debe revisar evidencia, limitaciones y contexto antes de cualquier acción externa. Para convertir este ejercicio en un sistema real harían falta datos oportunos de solicitudes, espera, flota, ocupación, tráfico, costos, aceptación y resultados, además de evaluación causal y gobernanza.

### Actividades de extensión

1. Cambiar `minimo_referencias` y explicar cómo afecta abstención y confianza.
2. Proponer una política que no dependa solo del volumen histórico y justificar qué datos nuevos necesita.
3. Agregar una decisión humana `ACEPTAR` o `RECHAZAR` sin convertirla automáticamente en evidencia de éxito.
4. Clasificar nuevamente el ambiente si el agente pasara de recomendar a ejecutar traslados.
5. Identificar qué parte del código implementa sensor, estado interno, función de agente y actuador.